# Identity Resolution
Cross-platform identity matching across CRM, GA4, Email, and Ad platforms.

The `IdentityResolver` builds a probabilistic identity graph by linking
records via:
1. **Exact email match** (weight 1.0) between CRM contacts and email recipients
2. **GA4 user-ID heuristic** derived from hashed email
3. **Campaign / timestamp proximity** for ad conversion matching


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.4f}'.format)

from datetime import datetime
from config import BASE_DIR, IDENTITY_CONFIG

print("Identity resolution config:")
for k, v in IDENTITY_CONFIG.items():
    print(f"  {k}: {v}")


In [ ]:
from src.extractors import (CRMExtractor, GA4Extractor,
                             EmailPlatformExtractor, MetaAdsExtractor, GoogleAdsExtractor)

END   = datetime(2025, 3, 1)
START = datetime(2025, 1, 1)

crm_ext   = CRMExtractor({"accounts_count": 80, "contacts_count": 200,
                           "opportunities_count": 40, "activities_count": 300})
ga4_ext   = GA4Extractor({"sessions_count": 800, "events_count": 3200})
email_ext = EmailPlatformExtractor({"campaigns": 5, "sends_count": 600})
meta_ext  = MetaAdsExtractor({"campaigns": 3})
gads_ext  = GoogleAdsExtractor({"campaigns": 3})

crm_raw   = crm_ext.extract(START, END)
ga4_raw   = ga4_ext.extract(START, END)
email_raw = email_ext.extract(START, END)
meta_raw  = meta_ext.extract(START, END)
gads_raw  = gads_ext.extract(START, END)

contacts  = crm_ext.contacts_df.copy()
sessions  = ga4_raw[ga4_raw['record_type'] == 'session'].copy()             if 'record_type' in ga4_raw.columns else ga4_raw.copy()

print(f"CRM contacts  : {len(contacts):,}")
print(f"GA4 sessions  : {len(sessions):,}")
print(f"Email sends   : {len(email_raw):,}")
print(f"Meta ad rows  : {len(meta_raw):,}")
print(f"Google ad rows: {len(gads_raw):,}")


In [ ]:
from src.transformers import IdentityResolver

resolver = IdentityResolver()

print("Running identity resolution…")
identity_graph = resolver.transform(
    crm_contacts      = contacts,
    ga4_sessions      = sessions,
    email_recipients  = email_raw,
    meta_conversions  = meta_raw,
    google_conversions= gads_raw,
)

print(f"\nIdentity graph rows: {len(identity_graph):,}")
print(f"Columns: {identity_graph.columns.tolist()}")
print()
display(identity_graph.head(8))


In [ ]:
# ── Match rates by method ─────────────────────────────────────────────────
total = len(identity_graph)

ga4_matched    = identity_graph['ga4_user_id'].notna().sum()
email_matched  = identity_graph['email_recipient_id'].notna().sum()
meta_matched   = identity_graph['meta_conversion_id'].notna() .sum()                  if 'meta_conversion_id' in identity_graph.columns else 0
google_matched = identity_graph['google_conversion_id'].notna().sum()                  if 'google_conversion_id' in identity_graph.columns else 0
unmatched      = identity_graph['review_flag'].sum() if 'review_flag' in identity_graph.columns else 0

match_rates = pd.DataFrame([
    {"Platform": "GA4",              "Matched": ga4_matched,    "Match Rate (%)": ga4_matched    / total * 100},
    {"Platform": "Email",            "Matched": email_matched,  "Match Rate (%)": email_matched  / total * 100},
    {"Platform": "Meta Ads",         "Matched": meta_matched,   "Match Rate (%)": meta_matched   / total * 100},
    {"Platform": "Google Ads",       "Matched": google_matched, "Match Rate (%)": google_matched / total * 100},
    {"Platform": "Unmatched (flag)", "Matched": unmatched,      "Match Rate (%)": unmatched      / total * 100},
])
print(f"=== Match Rates (CRM base: {total:,} contacts) ===")
display(match_rates.set_index("Platform").round(1))


In [ ]:
# ── Confidence distribution bar chart ─────────────────────────────────────
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.01]
labels = ['0-20%', '20-40%', '40-60%', '60-80%', '80-100%', '100%']
identity_graph['conf_bin'] = pd.cut(
    identity_graph['match_confidence'], bins=bins, labels=labels, right=True, include_lowest=True
)
conf_dist = identity_graph['conf_bin'].value_counts().sort_index()

colors = ['#d32f2f', '#f57c00', '#fbc02d', '#388e3c', '#1976d2', '#7b1fa2']
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(conf_dist.index.astype(str), conf_dist.values, color=colors[:len(conf_dist)], edgecolor='white')
ax.bar_label(bars, padding=3)
ax.set_title("Identity Match Confidence Distribution", fontsize=13, fontweight='bold')
ax.set_xlabel("Confidence Band")
ax.set_ylabel("Number of Records")
threshold_line = IDENTITY_CONFIG.get('min_confidence_threshold', 0.5)
ax.axvline(x=2.5, color='red', linestyle='--', linewidth=1.5, label=f"Min threshold ({threshold_line})")
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "visuals" / "nb03_confidence_dist.png", dpi=100, bbox_inches='tight')
plt.show()
print("Plot saved.")


In [ ]:
# ── Unmatched records ─────────────────────────────────────────────────────
if 'review_flag' in identity_graph.columns:
    unmatched_df = identity_graph[identity_graph['review_flag'] == True].copy()
    print(f"=== Unmatched Records: {len(unmatched_df):,} ({len(unmatched_df)/total:.1%}) ===")
    print()
    print("These contacts could not be linked to any external platform.")
    print("Implications:")
    print("  - Attribution gaps: their journey cannot be tracked through paid channels")
    print("  - Email suppression lists may be incomplete")
    print("  - GA4 tracking may be missing for certain acquisition sources")
    print()
    print("Sample unmatched contacts:")
    show_cols = [c for c in ['master_id', 'crm_contact_id', 'crm_email',
                              'match_confidence', 'match_method'] if c in unmatched_df.columns]
    display(unmatched_df[show_cols].head(10))


In [ ]:
# ── Identity graph sample ─────────────────────────────────────────────────
print("=== Identity Graph Sample (first 10 rows, all columns) ===")
display(identity_graph.head(10))

print()
print("Match method breakdown:")
method_counts = identity_graph['match_method'].value_counts()
display(method_counts.rename("count").to_frame())

print()
print(f"Target match rate (config): {IDENTITY_CONFIG['target_match_rate']:.0%}")
actual_matched = (identity_graph['match_confidence'] >= IDENTITY_CONFIG['min_confidence_threshold']).sum()
print(f"Actual matched (above min confidence threshold): "
      f"{actual_matched:,} / {total:,} = {actual_matched/total:.1%}")
